# Reviewer comment 4 — training and validation loss curves for all five agents

**What this notebook produces.** `training_curves.json` and `figure_training_curves.png`:
per-epoch training and validation loss for every one of the five detection agents,
which is what Reviewer 4 asked for ("complete training/validation loss curves for all
agents to demonstrate convergence and avoid overfitting concerns").

**Design principle: do not change the training recipes.** The five agents were trained with
different optimisers, schedules and stopping rules, and the paper's reported accuracies
depend on those exact recipes. Rewriting them into one uniform loop would produce curves
that no longer correspond to the published models. So this notebook runs each original
training script unmodified and *captures the per-epoch losses those scripts already print*.
Only FreqNet needs a genuine patch, because it is the one script that prints training loss
but never computes validation loss (see Step 5).

| Agent | How its history is captured |
|---|---|
| Biometric-Quality | **Already recovered** — embedded in the released checkpoint, no retraining needed |
| Audio Forensics (ECAPA-TDNN) | Script already writes a per-epoch CSV log |
| Cross-Modal (Lip-Sync) | Parsed from its per-epoch stdout (train + val loss) |
| Visual (XceptionNet) | Parsed from its per-epoch log (train + val loss, both stages) |
| Audio (FreqNet) | Small patch to compute and print validation loss |

**Runtime.** Preprocessing dominates. On a Colab A100 budget roughly 3-5 h for preprocessing
the full PolyGlotFake set and 2-4 h to train all five agents. Set `RUNTIME > Change runtime
type > GPU` before starting, and prefer a high-RAM instance.

---


## Step 1 — GPU check and repository


In [ ]:
!nvidia-smi
import torch, platform
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0),
          f'| {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GiB')


In [ ]:
%cd /content
!git clone https://github.com/saoirsebarry/multiagent-deepfake-detection.git repo 2>/dev/null || echo 'already cloned'
%cd /content/repo
!pip -q install albumentations speechbrain timm librosa opencv-python-headless facenet-pytorch
# NOTE: requirements.txt pins torch/torchvision/torchaudio/numpy. Colab already ships
# CUDA-matched builds (torch 2.11.0+cu128 on a T4), so installing those pins churns or
# breaks them. Install only what Colab is missing.


## Step 2 — PolyGlotFake

PolyGlotFake is distributed by its authors (Hou et al., arXiv:2405.08838). Point
`RAW_REAL` and `RAW_FAKE` at the extracted video directories. Mounting Drive is usually
the least painful route because Colab instances are ephemeral and the download is large.


In [ ]:
# One dataset root, not separate real/fake paths: the preprocessing script reads
# json_file/, real/ and fake/ beneath a single directory.
RAW_ROOT  = '/content/polyglot_lang'                      # extracted PolyGlotFake
PROCESSED = '/content/drive/MyDrive/polyglotfake/processed'  # persisted across sessions

import os
os.makedirs(PROCESSED, exist_ok=True)
for sub in ('json_file', 'real', 'fake'):
    p = os.path.join(RAW_ROOT, sub)
    print(f'  {p:52s} {"OK" if os.path.isdir(p) else "MISSING"}')
assert os.path.isdir(RAW_ROOT), f'extract PolyGlotFake to {RAW_ROOT} first'


## Step 3 — Preprocess

MTCNN face detection at confidence 0.95, frame stride 10, audio via librosa at 16 kHz
standardised to 5 s. Writes one `.npz` per clip. **Skip this cell if `PROCESSED` is already
populated from a previous session** — it is the slowest step by a wide margin.


In [ ]:
import os, shutil

# Free Drive is the binding constraint. Measured per-clip cost is ~3.8-4.5 MiB
# (20 face crops at 299x299 uint8, plus a float32 waveform), so:
#     train 1,052 + val 236 = 1,288 clips  ->  ~4.7-5.7 GiB   fits
#     test  2,162 clips                    ->  ~7.9-9.5 GiB   does not, and is not needed
# No training script reads test/, and the published test scores are already in
# paper_artifacts/source_csvs/. So write train and val only.
SPLITS = 'train,val'
BATCH  = 300     # clips per run; re-running resumes where it stopped
import os
WORKERS = max(1, (os.cpu_count() or 2) // 2)   # 1 = original serial loop
print(f'{os.cpu_count()} vCPUs -> --workers {WORKERS}')

free = shutil.disk_usage(os.path.dirname(PROCESSED.rstrip('/'))).free / 2**30
print(f'free on the output volume: {free:.2f} GiB')
assert free > 6.0, (f'only {free:.2f} GiB free. train+val needs ~6 GiB. Free space in Drive, or point PROCESSED at /content (local disk, lost when the session ends).')


In [ ]:
# Resumable: clips already written are skipped, so run this cell repeatedly until
# it reports nothing left. --min_free_gib aborts before the volume fills.
!python src/data_preprocessing/preprocessed_all_unbalanced.py \
    --data_dir     "$RAW_ROOT" \
    --output_dir   "$PROCESSED" \
    --splits       "$SPLITS" \
    --limit        $BATCH \
    --min_free_gib 1.0 \
    --workers      $WORKERS


In [ ]:
import os
counts = {s: (len(os.listdir(os.path.join(PROCESSED, s)))
              if os.path.isdir(os.path.join(PROCESSED, s)) else 0)
          for s in ('train', 'val')}
TARGET = {'train': 1052, 'val': 236}          # Table 3 of the manuscript
for s, n in counts.items():
    print(f'  {s:5s} {n:5d} / {TARGET[s]}  ({100*n/TARGET[s]:5.1f}%)')
done = sum(counts.values()); total = sum(TARGET.values())
used = sum(os.path.getsize(os.path.join(PROCESSED, s, f))
           for s in counts for f in os.listdir(os.path.join(PROCESSED, s))) / 2**30 \
       if done else 0.0
print(f'\n{done}/{total} clips, {used:.2f} GiB used'
      + (f', ~{used/done*(total-done):.2f} GiB still to write' if done else ''))
print('COMPLETE - go to Step 3b' if done >= total else
      'not finished - run the preprocessing cell again')


## Step 3b — Verify the regenerated split matches the published one

**Do not skip this.** Retrained agents are only comparable to the published ones if they
were trained on the same data. The split is deterministic by construction —
`preprocessed_all_unbalanced.py` sorts each language's file list (`lang_real_files.sort()`)
and splits it with `random_state=42` — so re-running it *should* reproduce the original
folds exactly. But that holds only if your copy of PolyGlotFake contains the same source
videos: the script silently skips any file that is missing (`if os.path.exists(...)`), and a
single absent video shifts every subsequent split boundary for that language.

This is checkable rather than assumable, because the released
`analysis_results_with_5_agents.csv` lists all 2,162 test clips by name. If the regenerated
`test/` directory matches that list exactly, your folds are the published folds and the
retrained curves describe the published models.


In [ ]:
import os, pandas as pd

published = set(pd.read_csv('paper_artifacts/source_csvs/'
                            'analysis_results_with_5_agents.csv').filepath)
regenerated = {f for f in os.listdir(os.path.join(PROCESSED, 'test')) if f.endswith('.npz')}

missing = published - regenerated      # in the paper's test set, absent from yours
extra   = regenerated - published      # in yours, not in the paper's
print(f'published test clips : {len(published)}')
print(f'regenerated test clips: {len(regenerated)}')
print(f'missing from yours    : {len(missing)}')
print(f'extra in yours        : {len(extra)}')

if not missing and not extra:
    print('\nSPLIT MATCHES EXACTLY - retrained agents are comparable to the published ones.')
else:
    print('\nSPLIT DIFFERS. Before trusting any retrained curve, check:')
    print('  1. whether your PolyGlotFake copy is complete (the script skips missing files)')
    for label, s in [('missing', missing), ('extra', extra)]:
        for f in sorted(s)[:5]: print(f'       {label}: {f}')
    print('  2. CRITICALLY - whether any published TEST clip is now in your train split,')
    print('     which would contaminate any comparison against the published numbers:')
    train = {f for f in os.listdir(os.path.join(PROCESSED, 'train')) if f.endswith('.npz')}
    leaked = published & train
    print(f'       published test clips now in train: {len(leaked)}')
    assert not leaked, 'published test clips leaked into training - do not use these curves'


## Step 4 — Recover the Biometric-Quality history (no retraining)

This agent's `Trainer` persisted its `history` dict inside the checkpoint, so its complete
curves are already in the repository. Running this cell reproduces the values used in the
revised manuscript and lets you confirm them independently.


In [ ]:
import torch, json
ck = torch.load('checkpoints/biometric/fine_tuning/best_model.pth',
                map_location='cpu', weights_only=False)
h = ck['history']
biometric = {'agent': 'Biometric-Quality',
             'source': 'released checkpoint (fine-tuning stage)',
             'epochs': list(range(1, len(h['train_loss']) + 1)),
             'train_loss': [float(x) for x in h['train_loss']],
             'val_loss':   [float(x) for x in h['val_loss']],
             'train_auc':  [float(x) for x in h['train_auc']],
             'val_auc':    [float(x) for x in h['val_auc']],
             'best_epoch': int(ck['epoch'])}
print(json.dumps(biometric, indent=2)[:900])
print('\nval loss monotonically decreasing after epoch 1:',
      all(biometric['val_loss'][i] >= biometric['val_loss'][i+1]
          for i in range(1, len(biometric['val_loss']) - 1)))


## Step 5 — Wire the data paths (do not skip)

**The five training scripts do not take a common data flag.** Verified against the
repository source: two of them accept a flag, one takes none at all, and two hard-code a
relative path. `docs/REPRODUCE.md` shows `--data_dir` for all five, which is wrong and will
fail. The real contract is:

| Script | How it finds data | Path it expects |
|---|---|---|
| `visual_xception.py` | hard-coded, relative to cwd | `polyglot_processed_all_unbalanced/` |
| `cross_modal_lipsync.py` | hard-coded, relative to cwd | `polyglot_processed_all_unbalanced/` |
| `audio_forensics_ecapa.py` | no flag, internal `CONFIG` | `data/polyglot_processed_all_unbalanced/` |
| `audio_freqnet.py` | `--dataroot` | any |
| `biometric_quality.py` | `--data_dir` | any |

Rather than edit five scripts, satisfy both hard-coded conventions with symlinks from the
repository root. Every script then works unmodified. Each expects `train/` and `val/`
subdirectories, which is what the preprocessing script writes.


In [ ]:
import os, pathlib
os.chdir('/content/repo')

for link in ['polyglot_processed_all_unbalanced', 'data/polyglot_processed_all_unbalanced']:
    p = pathlib.Path(link)
    p.parent.mkdir(parents=True, exist_ok=True)
    if p.is_symlink() or p.exists():
        p.unlink(missing_ok=True)
    p.symlink_to(PROCESSED)
    print(f'{link:44s} -> {os.path.realpath(link)}')

for link in ['polyglot_processed_all_unbalanced', 'data/polyglot_processed_all_unbalanced']:
    for split in ['train', 'val']:
        d = os.path.join(link, split)
        n = len(os.listdir(d)) if os.path.isdir(d) else 0
        print(f'  {d:52s} {n} files')
        assert n > 0, f'{d} is empty - preprocessing (Step 3) has not produced this split'


## Step 6 — Patch FreqNet so it also records validation loss

`audio_freqnet.py` is the only script that prints training loss but never computes a
validation loss: its validation loop accumulates predictions for accuracy only. Three exact
edits fix that, using the `criterion` (`nn.BCEWithLogitsLoss`) already in scope and the same
call the training loop makes. Nothing about the optimiser, schedule, augmentation or
stopping rule changes, so the trained model is unaffected.

The cell is idempotent and asserts each edit landed, so it fails loudly rather than
training for an hour and producing an unparseable log.


In [ ]:
import pathlib, py_compile, re
p = pathlib.Path('src/agents/audio_freqnet.py')
s = p.read_text()

if 'val_loss_total' in s:
    print('already patched')
else:
    before = s
    # 1. initialise the accumulator alongside the existing prediction buffers
    s = s.replace('        all_preds, all_labels = [], []',
                  '        all_preds, all_labels = [], []\n        val_loss_total = 0.0', 1)
    # 2. accumulate loss inside the existing no-grad validation loop
    s = s.replace('                outputs = model(specs)\n                preds = (outputs > 0.0).float()',
                  '                outputs = model(specs)\n'
                  '                val_loss_total += criterion(outputs, labels).item()\n'
                  '                preds = (outputs > 0.0).float()', 1)
    # 3. report it on the line the parser reads
    s = s.replace('| Avg Train Loss: {avg_train_loss:.4f} | Val Accuracy: {val_accuracy:.4f}',
                  '| Avg Train Loss: {avg_train_loss:.4f} '
                  '| Val Loss: {val_loss_total/max(len(val_loader),1):.4f} '
                  '| Val Accuracy: {val_accuracy:.4f}', 1)
    assert s != before, 'no edit applied - the source has changed shape, patch by hand'
    p.write_text(s)

assert s.count('val_loss_total') == 3, f"expected 3 references, found {s.count('val_loss_total')}"
py_compile.compile(str(p), doraise=True)
print('patch applied and file compiles')
print([l.strip() for l in s.splitlines() if 'val_loss_total' in l])


## Step 7 — Train the four remaining agents

Each run is teed to its own log. The commands below use each script's real interface as
established in Step 5 — note that ECAPA takes **no** data flag and Xception and Cross-Modal
take none either, relying on the symlinks.


In [ ]:
import pathlib
LOGS = pathlib.Path('/content/drive/MyDrive/polyglotfake/training_logs')
LOGS.mkdir(parents=True, exist_ok=True)
%cd /content/repo
print('logs ->', LOGS)


In [ ]:
# XceptionNet: two-stage transfer learning. No CLI flag - reads ./polyglot_processed_all_unbalanced
!python src/agents/visual_xception.py 2>&1 | tee "$LOGS/xception.log"


In [ ]:
# FreqNet: takes --dataroot (50 epochs, OneCycleLR, early stopping patience 10)
!python src/agents/audio_freqnet.py --dataroot polyglot_processed_all_unbalanced 2>&1 \
  | tee "$LOGS/freqnet.log"


In [ ]:
# Cross-Modal lip-sync: no CLI flag - reads ./polyglot_processed_all_unbalanced
!python src/agents/cross_modal_lipsync.py 2>&1 | tee "$LOGS/crossmodal.log"


In [ ]:
# ECAPA-TDNN head: no data flag at all - reads data/polyglot_processed_all_unbalanced from CONFIG
!python src/agents/audio_forensics_ecapa.py 2>&1 | tee "$LOGS/ecapa.log"
!find . -name '*.csv' -newermt '-6 hours' | head


In [ ]:
# Biometric-Quality: takes --data_dir. Only needed if you want to regenerate its curves;
# they are already recovered from the released checkpoint in Step 4.
# !python src/agents/biometric_quality.py --data_dir polyglot_processed_all_unbalanced 2>&1 \
#   | tee "$LOGS/biometric.log"


## Step 7 — Parse every log into one history file


In [ ]:
import re, json, glob, pathlib
import pandas as pd

def parse_log(path, patterns):
    """Return {field: [per-epoch values]} for the first pattern that matches."""
    text = pathlib.Path(path).read_text(errors='ignore')
    for pat, fields in patterns:
        hits = re.findall(pat, text)
        if hits:
            out = {f: [] for f in fields}
            for h in hits:
                for f, v in zip(fields, h if isinstance(h, tuple) else (h,)):
                    out[f].append(float(v))
            return out
    return {}

N = r'([0-9.]+)'
SPECS = {
  'Visual (XceptionNet)': ('xception.log', [
      (rf'Train Loss: {N}, Acc: {N} \| Val Loss: {N}, Acc: {N}',
       ['train_loss', 'train_acc', 'val_loss', 'val_acc'])]),
  'Audio (FreqNet)': ('freqnet.log', [
      (rf'Avg Train Loss: {N} \| Val Loss: {N} \| Val Accuracy: {N}',
       ['train_loss', 'val_loss', 'val_acc']),
      (rf'Avg Train Loss: {N} \| Val Accuracy: {N}',
       ['train_loss', 'val_acc'])]),
  'Cross-Modal (Lip-Sync)': ('crossmodal.log', [
      (rf'Train Loss: {N}, Train Acc: {N} \| Val Loss: {N}, Val Acc: {N}',
       ['train_loss', 'train_acc', 'val_loss', 'val_acc'])]),
  'Audio Forensics (ECAPA-TDNN)': ('ecapa.log', [
      (rf'Train Loss: {N}, Train AUC: {N}\s*\nVal Loss: {N}, Val AUC: {N}, Val Acc: {N}',
       ['train_loss', 'train_auc', 'val_loss', 'val_auc', 'val_acc'])]),
}

history = {'Biometric-Quality': biometric}
for agent, (fname, pats) in SPECS.items():
    f = LOGS / fname
    if not f.exists():
        print(f'MISSING {f} - skipping {agent}'); continue
    d = parse_log(f, pats)
    if not d:
        print(f'no epochs parsed from {f} for {agent}'); continue
    n = len(next(iter(d.values())))
    history[agent] = {'agent': agent, 'source': str(f),
                      'epochs': list(range(1, n + 1)), **d}
    print(f'{agent:32s} {n:3d} epochs  fields={list(d)}')

# ECAPA also writes a native CSV; prefer it when present
for c in glob.glob('**/*forensic*log*.csv', recursive=True) + glob.glob('**/training_log*.csv', recursive=True):
    df = pd.read_csv(c)
    if {'train_loss', 'val_loss'} <= set(df.columns):
        history['Audio Forensics (ECAPA-TDNN)'] = {
            'agent': 'Audio Forensics (ECAPA-TDNN)', 'source': c,
            'epochs': [int(e) for e in df['epoch']],
            'train_loss': [float(x) for x in df['train_loss']],
            'val_loss': [float(x) for x in df['val_loss']],
            'val_auc': [float(x) for x in df.get('val_auc', [])]}
        print('ECAPA history taken from native CSV:', c)
        break

json.dump(history, open('/content/training_curves.json', 'w'), indent=2)
print('\nagents captured:', list(history))


## Step 8 — Convergence and overfitting summary

This is the table that answers the reviewer directly: for each agent, the epoch at which
validation loss bottoms out, whether it subsequently rises (the overfitting signature), and
the final train/validation gap.


In [ ]:
import pandas as pd
rows = []
for a, h in history.items():
    tl, vl = h.get('train_loss', []), h.get('val_loss', [])
    if not vl:
        rows.append({'Agent': a, 'Epochs': len(tl), 'Best val epoch': 'n/a',
                     'Min val loss': 'n/a', 'Val loss rises after best': 'n/a',
                     'Final train-val gap': 'n/a'}); continue
    b = int(min(range(len(vl)), key=lambda i: vl[i]))
    after = vl[b + 1:]
    rows.append({'Agent': a, 'Epochs': len(vl), 'Best val epoch': b + 1,
                 'Min val loss': round(vl[b], 4),
                 'Val loss rises after best': 'yes' if after and max(after) > vl[b] * 1.05 else 'no',
                 'Final train-val gap': round(tl[-1] - vl[-1], 4) if tl else 'n/a'})
summary = pd.DataFrame(rows)
display(summary)
summary.to_csv('/content/convergence_summary.csv', index=False)


## Step 9 — The figure for the manuscript


In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 9, 'axes.spines.top': False,
                            'axes.spines.right': False, 'figure.dpi': 160})

agents = [a for a in ['Visual (XceptionNet)', 'Audio (FreqNet)',
                      'Cross-Modal (Lip-Sync)', 'Biometric-Quality',
                      'Audio Forensics (ECAPA-TDNN)'] if a in history]
fig, axes = plt.subplots(1, len(agents), figsize=(3.1 * len(agents), 2.9), sharey=False)
if len(agents) == 1: axes = [axes]

for ax, a in zip(axes, agents):
    h = history[a]
    ep = h['epochs']
    if h.get('train_loss'):
        ax.plot(ep, h['train_loss'], 'o-', ms=3, lw=1.4, color='#2b6cb0', label='train')
    if h.get('val_loss'):
        ax.plot(ep, h['val_loss'], 's--', ms=3, lw=1.4, color='#c05621', label='validation')
        b = int(min(range(len(h['val_loss'])), key=lambda i: h['val_loss'][i]))
        ax.axvline(ep[b], color='grey', lw=0.8, ls=':')
        ax.annotate(f'best ep {ep[b]}', xy=(ep[b], h['val_loss'][b]),
                    xytext=(3, 12), textcoords='offset points', fontsize=7, color='grey')
    ax.set_title(a.replace(' (', '\n('), fontsize=8.5)
    ax.set_xlabel('epoch')
    ax.legend(frameon=False, fontsize=7.5)
axes[0].set_ylabel('loss')
fig.tight_layout()
fig.savefig('/content/figure_training_curves.png', dpi=320, bbox_inches='tight')
fig.savefig('/content/figure_training_curves.pdf', bbox_inches='tight')
plt.show()


## Step 10 — Download the three artefacts

Send `training_curves.json`, `convergence_summary.csv` and `figure_training_curves.png`
back and they go straight into the revision as the new figure and the completed Table 14.


In [ ]:
from google.colab import files
for f in ['/content/training_curves.json', '/content/convergence_summary.csv',
          '/content/figure_training_curves.png', '/content/figure_training_curves.pdf']:
    try: files.download(f)
    except Exception as e: print('skip', f, e)
